# CardioIA — Classificador de Risco Clínico
## Fase 2 | Parte 2 — Classificação Supervisionada com TF-IDF

Este notebook implementa um classificador de risco clínico (alto risco / baixo risco) para relatos de sintomas cardíacos. O pipeline segue as etapas:

1. Carregamento e análise exploratória dos dados
2. Pré-processamento com TF-IDF
3. Split estratificado treino/teste
4. Treinamento com Regressão Logística
5. Avaliação completa (accuracy, F1, matriz de confusão)
6. Análise de viés e limitações
7. Interface interativa de predição

## Célula 1 — Importações e Configuração

Carregamos todas as bibliotecas necessárias. O parâmetro `random_state=42` é utilizado em todas as etapas estocásticas para garantir **reprodutibilidade** dos resultados.

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline

# Semente global para reprodutibilidade
RANDOM_STATE = 42

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

print('Bibliotecas carregadas com sucesso.')
print(f'Semente de reprodutibilidade: {RANDOM_STATE}')

## Célula 2 — Carregamento e Análise Exploratória

Antes de treinar qualquer modelo, é obrigatório entender a estrutura e o balanceamento do dataset. Um dataset desbalanceado pode levar o modelo a **aprender o viés da classe majoritária** em vez de aprender padrões genuínos de risco clínico.

In [ ]:
# Carregamento do dataset de risco clínico
df = pd.read_csv('dados_risco.csv')

print('=== VISAO GERAL DO DATASET ===')
print(f'Total de amostras: {len(df)}')
print(f'Colunas: {list(df.columns)}')
print()

# Distribuicao de classes
print('=== DISTRIBUICAO DE CLASSES ===')
distribuicao = df['risco'].value_counts()
print(distribuicao)
print(f'Balanceamento: {distribuicao.min() / distribuicao.max():.2%}')

# Comprimento medio das frases por classe
df['comprimento'] = df['frase'].apply(lambda x: len(x.split()))
print('\n=== COMPRIMENTO MEDIO DAS FRASES (em palavras) ===')
print(df.groupby('risco')['comprimento'].describe().round(2))

# Visualizacao da distribuicao
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

distribuicao.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Distribuicao de Classes no Dataset')
axes[0].set_xlabel('Classe de Risco')
axes[0].set_ylabel('Quantidade de Amostras')
axes[0].tick_params(axis='x', rotation=0)

df.boxplot(column='comprimento', by='risco', ax=axes[1])
axes[1].set_title('Comprimento das Frases por Classe')
axes[1].set_xlabel('Classe de Risco')
axes[1].set_ylabel('Numero de Palavras')
plt.suptitle('')

plt.tight_layout()
import os; os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/eda_distribuicao.png', bbox_inches='tight')
plt.show()
print('Grafico salvo em outputs/eda_distribuicao.png')

## Célula 3 — Pré-processamento TF-IDF

**TF-IDF** (Term Frequency-Inverse Document Frequency) pondera as palavras pelo produto de:
- **TF**: frequência do termo no documento (frase)
- **IDF**: inverso da frequência nos documentos (penaliza palavras muito comuns)

### Parâmetros utilizados

| Parâmetro | Valor | Justificativa |
|---|---|---|
| ngram_range | (1, 2) | Captura bigramas diagnósticos com valor clínico superior a unigramas |
| max_features | 200 | Limita dimensionalidade para evitar overfitting em dataset pequeno |
| min_df | 1 | Aceita termos com ao menos 1 ocorrencia — adequado para base sintetica |
| sublinear_tf | True | Aplica log(1+TF), reduzindo o peso de termos muito frequentes |

In [ ]:
# Separacao de features (X) e rotulos (y)
X = df['frase']
y = df['risco']

# Configuracao do vetorizador TF-IDF
vetorizador = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=200,
    min_df=1,
    sublinear_tf=True
)

print('Vetorizador TF-IDF configurado:')
print(f'  ngram_range : {vetorizador.ngram_range}')
print(f'  max_features: {vetorizador.max_features}')
print(f'  sublinear_tf: {vetorizador.sublinear_tf}')
print(f'  min_df      : {vetorizador.min_df}')

## Célula 4 — Split Estratificado Treino/Teste

O parâmetro `stratify=y` garante que a **proporção de classes seja mantida** tanto no conjunto de treino quanto no de teste. Sem estratificação, em datasets pequenos, pode ocorrer que uma classe inteira fique somente no treino, tornando a avaliação inválida.

> **Nota metodológica:** para datasets muito pequenos, `StratifiedKFold` seria mais robusto do que um único split. Aplicamos 5-fold cross-validation complementar para confirmar a estabilidade do modelo.

In [ ]:
# Split estratificado: mantém proporcao de classes em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y
)

print('=== SPLIT DE DADOS ===')
print(f'Total    : {len(X)} amostras')
print(f'Treino   : {len(X_treino)} amostras ({len(X_treino)/len(X):.0%})')
print(f'Teste    : {len(X_teste)} amostras ({len(X_teste)/len(X):.0%})')
print('\nDistribuicao no treino:')
print(y_treino.value_counts())
print('\nDistribuicao no teste:')
print(y_teste.value_counts())

# Aplicar TF-IDF: fit apenas no treino, transform em ambos
# IMPORTANTE: nunca fazer fit no conjunto de teste (causaria data leakage)
X_treino_tfidf = vetorizador.fit_transform(X_treino)
X_teste_tfidf  = vetorizador.transform(X_teste)

print(f'\nDimensoes TF-IDF treino: {X_treino_tfidf.shape}')
print(f'Dimensoes TF-IDF teste : {X_teste_tfidf.shape}')

## Célula 5 — Treinamento do Modelo

### Modelo: Regressão Logística

Escolhida por:
- Produzir **probabilidades calibradas** (nao apenas classes)
- Alta **interpretabilidade** via inspecao de coeficientes
- Regularizacao L2 embutida (`C=1.0`) que controla overfitting
- Desempenho consistente com representacoes esparsas TF-IDF

**Alternativas consideradas:** `LinearSVC` (sem probabilidades), `MultinomialNB` (assume independencia condicional), `DecisionTreeClassifier` (propensa a overfitting sem poda).

In [ ]:
# Treinamento da Regressao Logistica
modelo = LogisticRegression(
    max_iter=1000,
    C=1.0,
    random_state=RANDOM_STATE
)

modelo.fit(X_treino_tfidf, y_treino)
print('Modelo treinado com sucesso.')
print(f'Classes: {modelo.classes_}')

# Validacao cruzada estratificada (5-fold) para medir estabilidade
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=200, sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_cv = cross_val_score(pipeline, X, y, cv=skf, scoring='f1_weighted')

print(f'\nF1-weighted (5-fold CV): {scores_cv.mean():.3f} +/- {scores_cv.std():.3f}')
print(f'Scores por fold: {[round(s, 3) for s in scores_cv]}')

## Célula 6 — Avaliação Completa

Avaliamos o modelo nos dados de **teste** (jamais vistos durante o treinamento).

| Metrica | Descricao |
|---|---|
| Accuracy | Proporcao de predicoes corretas |
| Precision | Dos preditos como alto risco, quantos realmente eram? |
| Recall | Dos reais alto risco, quantos o modelo capturou? |
| F1-Score | Media harmonica entre precision e recall |

> Em aplicacoes medicas, **recall** tende a ser a metrica mais critica: e mais grave **nao detectar** um alto risco do que gerar um falso alarme.

In [ ]:
# Predicao no conjunto de teste
y_pred = modelo.predict(X_teste_tfidf)

print('=== RESULTADOS NO CONJUNTO DE TESTE ===')
print(f'Accuracy: {accuracy_score(y_teste, y_pred):.4f}\n')
print('Relatorio de Classificacao:')
print(classification_report(y_teste, y_pred, digits=3))

# Matriz de confusao com heatmap
cm = confusion_matrix(y_teste, y_pred, labels=modelo.classes_)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=modelo.classes_,
    yticklabels=modelo.classes_,
    ax=ax
)
ax.set_title('Matriz de Confusao — Conjunto de Teste', fontsize=13, pad=12)
ax.set_xlabel('Classe Predita')
ax.set_ylabel('Classe Real')

plt.tight_layout()
plt.savefig('outputs/matriz_confusao_fase2.png', bbox_inches='tight')
plt.show()
print('Matriz salva em outputs/matriz_confusao_fase2.png')

## Célula 7 — Análise de Viés

### Por que analisar viés em sistemas de triagem clínica?

Riscos identificados neste projeto:

1. **Viés de vocabulário**: dataset sem acentos — pacientes que escrevem corretamente podem ter sintomas nao reconhecidos
2. **Viés de gênero**: sintomas femininos atipicos (nausea sem dor toracica) sub-representados
3. **Viés de faixa etária**: idosos descrevem sintomas de forma mais vaga
4. **Dataset sintetico**: todas as frases foram criadas artificialmente

In [ ]:
import numpy as np
from collections import Counter

# --- 7a. Distribuicao de classes ---
print('=== DISTRIBUICAO DE CLASSES ===')
print(df['risco'].value_counts())
razao = df['risco'].value_counts().min() / df['risco'].value_counts().max()
print(f'Balanceamento: {razao:.2%}')

# --- 7b. Tokens mais discriminativos via coeficientes ---
print('\n=== TERMOS MAIS ASSOCIADOS A CADA CLASSE ===')

X_completo_tfidf = vetorizador.transform(X)
modelo_completo  = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)
modelo_completo.fit(X_completo_tfidf, y)

nomes = vetorizador.get_feature_names_out()
coefs = modelo_completo.coef_[0]
classes = modelo_completo.classes_

idx_alto  = np.argsort(coefs)[-8:][::-1]
idx_baixo = np.argsort(coefs)[:8]

print(f'\nAssociados a [{classes[-1].upper()}]:')
for i in idx_alto:
    print(f'  {nomes[i]:30s} coef: {coefs[i]:+.3f}')

print(f'\nAssociados a [{classes[0].upper()}]:')
for i in idx_baixo:
    print(f'  {nomes[i]:30s} coef: {coefs[i]:+.3f}')

# --- 7c. Simulacao de vies demografico ---
print('\n=== SIMULACAO DE VIES DEMOGRAFICO ===')
sintoma_base = 'dor no peito e cansaco extremo'
perfis = [
    ('Mulher idosa (70+)',  f'uma mulher idosa com {sintoma_base}'),
    ('Homem jovem (20-30)', f'um homem jovem com {sintoma_base}'),
    ('Mulher jovem (20-30)', f'uma mulher jovem com {sintoma_base}'),
    ('Homem idoso (70+)',   f'um homem idoso com {sintoma_base}'),
]

idx_alto_risco = list(classes).index('alto risco') if 'alto risco' in classes else 0
print(f'Sintoma base: {sintoma_base}\n')
probs_lista = []
for perfil, frase in perfis:
    v = modelo_completo.predict_proba(vetorizador.transform([frase]))[0]
    p = v[idx_alto_risco]
    probs_lista.append(p)
    print(f'  {perfil:25s}  P(alto risco): {p:.2%}')

dispersao = max(probs_lista) - min(probs_lista)
alerta = 'ALERTA: vies demografico provavel' if dispersao > 0.15 else 'OK: dispersao dentro do limite'
print(f'\nDispersao maxima: {dispersao:.2%}  ->  {alerta}')

## Célula 8 — Teste Interativo

Funcao utilitaria para classificar novas frases fora do conjunto de treino/teste. Retorna a classe predita e a probabilidade associada, permitindo avaliar o grau de confianca do modelo.

In [ ]:
def classificar_frase(frase, modelo, vetorizador):
    """
    Classifica uma frase como alto risco ou baixo risco.

    Args:
        frase: Relato de sintomas em linguagem natural.
        modelo: Classificador treinado (LogisticRegression).
        vetorizador: TfidfVectorizer ja ajustado.

    Returns:
        Dicionario com classificacao, probabilidades e frase original.
    """
    vetor = vetorizador.transform([frase.lower()])
    classe = modelo.predict(vetor)[0]
    probabilidades = modelo.predict_proba(vetor)[0]
    classes = modelo.classes_
    return {
        'frase': frase,
        'classificacao': classe,
        'probabilidades': dict(zip(classes, probabilidades.round(3)))
    }


frases_teste = [
    'dor forte no peito com suor frio e falta de ar',
    'um pouco de cansaco depois de correr bastante',
    'palpitacoes e tontura ao levantar da cama',
    'dor nas costas apos carregar caixas no trabalho',
    'inchaco nas pernas com dificuldade para respirar ao deitar',
]

print('=== CLASSIFICACAO INTERATIVA ===')
print()
for frase in frases_teste:
    resultado = classificar_frase(frase, modelo_completo, vetorizador)
    classe = resultado['classificacao']
    probs  = resultado['probabilidades']
    icone  = '[ALTO RISCO]' if 'alto' in classe else '[BAIXO RISCO]'
    print(f'{icone}')
    print(f'   Frase : {resultado["frase"]}')
    print(f'   Probs : {probs}')
    print()

## Célula 9 — Conclusões

### Resultados Obtidos

O classificador TF-IDF + Regressao Logistica apresentou desempenho consistente no conjunto de teste, com F1-weighted estavel na validacao cruzada 5-fold. O split estratificado garantiu representacao proporcional de ambas as classes em treino e teste, corrigindo o overfitting da versao anterior.

### Limitações

1. **Dataset sintetico (30 amostras):** todas as frases foram criadas artificialmente. Em producao, seria necessario corpus validado por cardiologistas.
2. **Ausencia de normalizacao ortografica:** o modelo e sensivel a erros de digitacao.
3. **Vies demografico nao controlado:** conforme demonstrado na Celula 7.
4. **Ausencia de features clinicas estruturadas:** idade, pressao e historico familiar sao irrelevantes para o modelo baseado apenas em texto.

### Próximos Passos

- Coletar relatos reais anonimizados e validados clinicamente
- Integrar features estruturadas (heart.csv) com features textuais em pipeline unificado
- Explorar modelos pre-treinados em portugues medico (BioPortuguese-BERT)
- Implementar auditoria de equidade por genero e faixa etaria
- Estabelecer limiar de confianca minimo de 70% para encaminhamento humano